In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
# from xgboost import XGBClassifier  # 트리 중요도 기반 피처 필터링을 LR 전처리에서 주석 처리하며 미사용
from lightgbm import LGBMClassifier

In [2]:
DATA_PATH = os.path.join("../data/", "santander-customer-satisfaction")

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))

print(f"Train 데이터 크기: {train_df.shape}")

train_df.head()

Train 데이터 크기: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [3]:
# 정답 라벨 분리 (맨 마지막 컬럼 TARGET)
y_labels = train_df.iloc[:, -1].copy()

# ID와 TARGET을 제외한 순수 피처 세트 분리
X_features = train_df.drop(columns=['ID', 'TARGET']).copy()

print(f"X_features Shape: {X_features.shape}")
print(f"y_labels Shape: {y_labels.shape}")

X_features Shape: (76020, 369)


y_labels Shape: (76020,)


In [4]:
# var3 결측성 이상치(-999999)를 최빈값(2)으로 치환
X_features['var3'] = X_features['var3'].replace(-999999, 2)

print("var3 이상치 치환 완료 (-999999 잔여 개수):", (X_features['var3'] == -999999).sum())

var3 이상치 치환 완료 (-999999 잔여 개수): 0


In [5]:
# var38뿐 아니라 왜도(|skew|>1)가 큰 금액/거래 피처 전반에 signed log1p 변환 적용
# (StandardScaler는 평균/분산만 맞출 뿐 극단적 왜도·이상치를 완화하지 못해
#  로지스틱 회귀(선형 모델)에서는 왜곡된 분포가 결정 경계에 과도한 영향을 줄 수 있음)
skew_vals = X_features.drop(columns=['var3']).skew()
high_skew_cols = skew_vals[skew_vals.abs() > 1].index.tolist()

for col in high_skew_cols:
    X_features[col] = np.sign(X_features[col]) * np.log1p(np.abs(X_features[col]))

print(f"로그 변환 적용 컬럼 수: {len(high_skew_cols)} / {X_features.shape[1]} (var38 포함)")

로그 변환 적용 컬럼 수: 326 / 369 (var38 포함)


In [6]:
# 고객별 0의 개수 카운트
X_features['n0'] = (X_features == 0).sum(axis=1)

# 고객별 거래/잔액의 표준편차
X_features['row_std'] = X_features.std(axis=1)

print("행 통계량 파생 변수(n0, row_std) 생성 완료")

행 통계량 파생 변수(n0, row_std) 생성 완료


In [7]:
# 1. 분산 0인 상수 컬럼 제거 <- 제거 한 것과 안 한 것 둘 다 결과 같음
zero_var_cols = [col for col in X_features.columns if X_features[col].nunique() == 1]
X_features.drop(columns=zero_var_cols, inplace=True)
print(f"1) 제거된 상수 컬럼 수: {len(zero_var_cols)}")

# 2. 중복 컬럼 초고속 제거
dup_cols = X_features.T.duplicated()
dup_col_names = X_features.columns[dup_cols].tolist()
X_features.drop(columns=dup_col_names, inplace=True)
print(f"2) 제거된 중복 컬럼 수: {len(dup_col_names)}")

# 3. var6 유사 피처 및 다중공선성 delta 컬럼 제거
manual_remove = [c for c in X_features.columns if 'var6' in c] + [
    'delta_imp_reemb_var13_1y3', 'delta_imp_reemb_var17_1y3', 
    'delta_imp_trasp_var17_in_1y3', 'delta_imp_trasp_var33_in_1y3'
]
manual_remove = [c for c in manual_remove if c in X_features.columns]
X_features.drop(columns=manual_remove, inplace=True)
print(f"3) 추가 제거된 유사/노이즈 컬럼 수: {len(manual_remove)}")



1) 제거된 상수 컬럼 수: 34


2) 제거된 중복 컬럼 수: 29
3) 추가 제거된 유사/노이즈 컬럼 수: 9


In [8]:
# 1차 분할: 학습 세트(80%)와 최종 테스트 세트(20%)로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels,
    test_size=0.2, 
    stratify=y_labels, 
    random_state=0
)

# 2차 분할: 학습 세트를 다시 훈련용(70%)과 조기 중단 감시용(30%)으로 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.3, 
    stratify=y_train, 
    random_state=0
)

print(f"훈련 세트(X_tr) Shape: {X_tr.shape}")
print(f"검증 세트(X_val) Shape: {X_val.shape}")
print(f"테스트 세트(X_test) Shape: {X_test.shape}")

훈련 세트(X_tr) Shape: (42571, 299)
검증 세트(X_val) Shape: (18245, 299)
테스트 세트(X_test) Shape: (15204, 299)


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# (1), (2) XGBoost 트리 중요도 기반 피처 필터링은 트리 분기 기준으로 산출되므로
# 로지스틱 회귀(선형 모델)의 피처 유용성 판단 기준과 맞지 않아 주석 처리함
# (트리에서 중요도 0이어도 선형 관계로는 유효한 피처일 수 있음)

# # (1) 훈련 데이터(X_tr)로만 베이스 모델 학습하여 피처 중요도 측정
# base_xgb = XGBClassifier(
#     n_estimators=100,
#     learning_rate=0.1,
#     random_state=156,
#     n_jobs=-1
# )
# base_xgb.fit(X_tr, y_tr)

# # (2) 중요도가 0인 피처 확인 후 3개 세트(X_tr, X_val, X_test)에서 동일하게 제거
# feat_imp = pd.Series(base_xgb.feature_importances_, index=X_tr.columns)
# zero_imp_cols = feat_imp[feat_imp == 0].index.tolist()

# X_tr = X_tr.drop(columns=zero_imp_cols)
# X_val = X_val.drop(columns=zero_imp_cols)
# X_test = X_test.drop(columns=zero_imp_cols)
# print(f"제거된 중요도 0인 컬럼 수: {len(zero_imp_cols)}")

# (3) X_tr 기준으로 StandardScaler 학습(fit) 후 변환(transform)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# (4) X_tr 기준으로 PCA 학습(fit) 후 주성분 2개 변환(transform)
pca = PCA(n_components=5, random_state=156)
pca_tr = pca.fit_transform(X_tr_scaled)
pca_val = pca.transform(X_val_scaled)
pca_test = pca.transform(X_test_scaled)

# (5) 각 세트에 pca1, pca2 컬럼 추가
X_tr['pca1'] = pca_tr[:, 0]
X_tr['pca2'] = pca_tr[:, 1]
X_tr['pca3'] = pca_tr[:, 2]
X_tr['pca4'] = pca_tr[:, 3]
X_tr['pca5'] = pca_tr[:, 4]

X_val['pca1'] = pca_val[:, 0]
X_val['pca2'] = pca_val[:, 1]
X_val['pca3'] = pca_val[:, 2]
X_val['pca4'] = pca_val[:, 3]
X_val['pca5'] = pca_val[:, 4]

X_test['pca1'] = pca_test[:, 0]
X_test['pca2'] = pca_test[:, 1]
X_test['pca3'] = pca_test[:, 2]
X_test['pca4'] = pca_test[:, 3]
X_test['pca5'] = pca_test[:, 4]

print(f"훈련 세트(X_tr) 최종 Shape: {X_tr.shape}")
print(f"검증 세트(X_val) 최종 Shape: {X_val.shape}")
print(f"테스트 세트(X_test) 최종 Shape: {X_test.shape}")


훈련 세트(X_tr) 최종 Shape: (42571, 304)
검증 세트(X_val) 최종 Shape: (18245, 304)
테스트 세트(X_test) 최종 Shape: (15204, 304)


In [10]:
from sklearn.linear_model import LogisticRegression

# LogisticRegression은 스케일에 민감하므로 X_tr 기준 StandardScaler로 재스케일링
lr_scaler = StandardScaler()
X_tr_lr = lr_scaler.fit_transform(X_tr)
X_val_lr = lr_scaler.transform(X_val)
X_test_lr = lr_scaler.transform(X_test)

lr_clf = LogisticRegression(max_iter=1000, random_state=0)
lr_clf.fit(X_tr_lr, y_tr)

val_pred = lr_clf.predict_proba(X_val_lr)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)
print(f"로지스틱 회귀 검증 AUC: {val_auc:.4f}")

test_pred = lr_clf.predict_proba(X_test_lr)[:, 1]
lr_roc_score = roc_auc_score(y_test, test_pred)

print("=" * 40)
print(f"로지스틱 회귀 최종 테스트 세트 ROC-AUC: {lr_roc_score:.4f}")
print("=" * 40)

로지스틱 회귀 검증 AUC: 0.8235
로지스틱 회귀 최종 테스트 세트 ROC-AUC: 0.7930


In [11]:
# 검증 세트(X_val) 기준으로 정규화 강도(C)와 클래스 불균형 보정(class_weight) 탐색
# TARGET 비율이 96:4로 불균형하므로 class_weight='balanced' 적용 여부도 함께 비교
# (1차 탐색에서 최적 C가 탐색 범위의 최소값(0.001)이라 더 작은 값까지 범위 확장)
param_candidates = [
    {'C': c, 'class_weight': cw}
    for c in [0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
    for cw in [None, 'balanced']
]

best_val_auc = -1
best_params = None
for params in param_candidates:
    clf = LogisticRegression(max_iter=1000, random_state=0, **params)
    clf.fit(X_tr_lr, y_tr)
    val_auc_i = roc_auc_score(y_val, clf.predict_proba(X_val_lr)[:, 1])
    if val_auc_i > best_val_auc:
        best_val_auc = val_auc_i
        best_params = params

print(f"최적 하이퍼파라미터: {best_params} (검증 AUC={best_val_auc:.4f})")

# 최적 파라미터로 재학습 후 최종 테스트 세트 AUC 산출
best_lr = LogisticRegression(max_iter=1000, random_state=0, **best_params)
best_lr.fit(X_tr_lr, y_tr)
test_pred = best_lr.predict_proba(X_test_lr)[:, 1]
lr_tuned_roc_score = roc_auc_score(y_test, test_pred)

print("=" * 40)
print(f"로지스틱 회귀(튜닝) 최종 테스트 세트 ROC-AUC: {lr_tuned_roc_score:.4f}")
print("=" * 40)

최적 하이퍼파라미터: {'C': 0.001, 'class_weight': 'balanced'} (검증 AUC=0.8301)


로지스틱 회귀(튜닝) 최종 테스트 세트 ROC-AUC: 0.7978
